# Export Lorenz Macro States

Load the saved `stage2_lorzen_macro` checkpoint, encode every state from `loc_data_lorzen/generated_data.npz`, apply one macro-dynamics step, and export `result/yt.csv` plus `result/yt+1.csv`.

In [1]:
from __future__ import annotations

import re
import sys
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union
import numpy as np
import pandas as pd
import torch
from IPython.display import display

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "src" / "models_macro.py").exists() and (candidate / "loc_model_stage2").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

from src.models_macro import Parellel_Renorm_Dynamic
torch.set_grad_enabled(False)

DEFAULT_RUN_NAME = "stage2_lorenz_macro"
DEFAULT_DATA_PATH = Path("loc_data_lorenz") / "generated_data.npz"
DEFAULT_MODEL_NAME = "model_scale1.pkl"
DEFAULT_SUMMARY_NAME = "summary_scale1.csv"
DEFAULT_OUTPUT_DIR = Path("result/lorenz")

In [2]:
def find_project_root(start: Optional[Path] = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "models_macro.py").exists() and (candidate / "loc_model_stage2").exists():
            return candidate
    raise FileNotFoundError("Could not locate the causal_network_mlp_tanh project root.")


def parse_int_list(value) -> List[int]:
    text = "" if value is None else str(value).strip()
    if text == "" or text.lower() == "nan":
        return []
    return [int(part.strip()) for part in text.split(",") if part.strip()]


def load_lorzen_data(data_path: Path) -> Tuple[np.ndarray, List[int]]:
    with np.load(data_path, allow_pickle=False) as archive:
        if "data" not in archive:
            raise ValueError(f"Missing 'data' array in: {data_path}")
        origin_data = np.asarray(archive["data"], dtype=np.float32)
        group = parse_int_list(",".join(str(int(value)) for value in archive["group"].reshape(-1))) if "group" in archive else []

    if origin_data.ndim != 2:
        raise ValueError(f"Expected Lorenz data with shape [T, N], but got {origin_data.shape}")
    if group and sum(group) != origin_data.shape[1]:
        raise ValueError(f"Group sizes must sum to {origin_data.shape[1]}, but got {group}")
    return origin_data, group


def load_state_dict(model_path: Path, device: torch.device):
    try:
        return torch.load(model_path, map_location=device, weights_only=True)
    except TypeError:
        return torch.load(model_path, map_location=device)


def infer_scale_dims(state_dict) -> List[int]:
    layers_by_scale: Dict[int, List[Tuple[int, torch.Tensor]]] = {}
    for name, tensor in state_dict.items():
        match = re.match(r"^dynamics_modules\.(\d+)\.(\d+)\.weight$", name)
        if match:
            layers_by_scale.setdefault(int(match.group(1)), []).append((int(match.group(2)), tensor))
    if not layers_by_scale:
        raise ValueError("The checkpoint does not contain macro dynamics modules.")
    return [int(max(layers_by_scale[idx], key=lambda item: item[0])[1].shape[0]) for idx in sorted(layers_by_scale)]


def count_weight_layers(state_dict, pattern: str) -> int:
    return sum(1 for name in state_dict if re.match(pattern, name))


def infer_checkpoint_config(state_dict, summary_row: Dict, group: List[int]) -> Dict:
    scale_dims = infer_scale_dims(state_dict)
    encoder_type = "invertible" if any(".group_flows." in name or ".flow.flow." in name for name in state_dict) else "mlp"

    encoder_weight = state_dict.get("group_transitions.0.encoders.0.0.weight")
    dynamics_weight = state_dict.get("dynamics_modules.0.0.weight")
    hidden_units1 = int(encoder_weight.shape[0]) if encoder_weight is not None else int(summary_row.get("hidden_units1", 100))
    hidden_units2 = int(dynamics_weight.shape[0]) if dynamics_weight is not None else int(summary_row.get("hidden_units2", 100))

    dynamics_num_layers = count_weight_layers(state_dict, r"^dynamics_modules\.0\.\d+\.weight$")
    if dynamics_num_layers == 0:
        dynamics_num_layers = int(summary_row.get("dynamics_num_layers", 4))

    flow_num_layers = count_weight_layers(state_dict, r"^group_transitions\.0\.encoders\.0\.\d+\.weight$")
    if flow_num_layers == 0:
        flow_num_layers = int(summary_row.get("flow_num_layers", 3))

    return {
        "logical_scale_id": int(summary_row.get("scale_id", 0)),
        "scale_dims": scale_dims,
        "latent_size": int(scale_dims[-1]),
        "hidden_units1": hidden_units1,
        "hidden_units2": hidden_units2,
        "flow_num_layers": flow_num_layers,
        "dynamics_num_layers": dynamics_num_layers,
        "reduce_dims": parse_int_list(summary_row.get("reduce_dims", "")),
        "group": group or parse_int_list(summary_row.get("group", "")),
        "encoder_type": encoder_type,
    }


def build_stage2_lorzen_model(
    num_nodes: int,
    model_path: Path,
    summary_path: Path,
    group: List[int],
    device: Union[str, torch.device] = "cpu",
):
    device = torch.device(device)
    summary_row = pd.read_csv(summary_path).iloc[0].to_dict()
    state_dict = load_state_dict(model_path, device=device)
    config = infer_checkpoint_config(state_dict, summary_row=summary_row, group=group)

    model = Parellel_Renorm_Dynamic(
        sym_size=int(num_nodes),
        latent_size=int(config["latent_size"]),
        effect_size=int(num_nodes),
        cut_size=2,
        hidden_units1=int(config["hidden_units1"]),
        hidden_units2=int(config["hidden_units2"]),
        normalized_state=True,
        device=device,
        is_random=False,
        flow_num_layers=int(config["flow_num_layers"]),
        dynamics_num_layers=int(config["dynamics_num_layers"]),
        decode_noise_scale=0.0,
        reduce_dims=config["reduce_dims"],
        group=config["group"],
        encoder_type=config["encoder_type"],
    ).to(device)
    model.load_state_dict(state_dict)
    model.eval()
    return model, config


def encode_and_predict_next_macro_state(
    model: Parellel_Renorm_Dynamic,
    origin_data: np.ndarray,
    logical_scale_id: int,
) -> Tuple[np.ndarray, np.ndarray]:
    device = next(model.parameters()).device
    source = torch.as_tensor(origin_data, dtype=torch.float32, device=device)
    with torch.no_grad():
        yt = model.encoding1(source, logical_scale_id)[logical_scale_id]
        yt_next = model._apply_dynamics(yt, logical_scale_id, inverse=False)
    return yt.cpu().numpy().astype(np.float32), yt_next.cpu().numpy().astype(np.float32)


def export_macro_states(
    project_root: Optional[Path] = None,
    run_name: str = DEFAULT_RUN_NAME,
    data_path: Path = DEFAULT_DATA_PATH,
    output_dir: Path = DEFAULT_OUTPUT_DIR,
    device: Union[str, torch.device] = "cpu",
) -> Tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    project_root = find_project_root(project_root)
    data_path = project_root / data_path
    model_path = project_root / "loc_model_stage2" / run_name / DEFAULT_MODEL_NAME
    summary_path = project_root / "loc_result_stage2" / run_name / DEFAULT_SUMMARY_NAME
    output_dir = project_root / output_dir
    
    origin_data, group = load_lorzen_data(data_path)
    model, config = build_stage2_lorzen_model(
        num_nodes=origin_data.shape[1],
        model_path=model_path,
        summary_path=summary_path,
        group=group,
        device=device,
    )
    scale_id = int(config["logical_scale_id"])
    if not 0 <= scale_id < len(config["scale_dims"]):
        raise ValueError(f"logical_scale_id={scale_id} is outside checkpoint scale_dims={config['scale_dims']}")

    yt, yt_next = encode_and_predict_next_macro_state(model, origin_data, scale_id)
    output_dir.mkdir(parents=True, exist_ok=True)
    yt_path = output_dir / "yt.csv"
    yt_next_path = output_dir / "yt+1.csv"
    pd.DataFrame(yt).to_csv(yt_path, index=False, header=False)
    pd.DataFrame(yt_next).to_csv(yt_next_path, index=False, header=False)

    summary = pd.DataFrame(
        [
            {
                "data_path": str(data_path.resolve()),
                "model_path": str(model_path.resolve()),
                "logical_scale_id": scale_id,
                "checkpoint_scale_dims": str(config["scale_dims"]),
                "encoder_type": config["encoder_type"],
                "num_samples": int(yt.shape[0]),
                "macro_dim": int(yt.shape[1]),
                "yt_csv": str(yt_path.resolve()),
                "yt_next_csv": str(yt_next_path.resolve()),
            }
        ]
    )
    return summary, yt, yt_next

In [3]:
PROJECT_ROOT = find_project_root()
EXPORT_SUMMARY, yt, yt_next = export_macro_states(project_root=PROJECT_ROOT)
EXPORT_SUMMARY

,data_path,model_path,logical_scale_id,checkpoint_scale_dims,encoder_type,num_samples,macro_dim,yt_csv,yt_next_csv
0,/home/wangzhipeng/code/causal_network/causal_n...,/home/wangzhipeng/code/causal_network/causal_n...,0,"[8, 4, 2, 1]",mlp,1000,8,/home/wangzhipeng/code/causal_network/causal_n...,/home/wangzhipeng/code/causal_network/causal_n...


In [4]:
preview = pd.concat(
    {
        "yt": pd.DataFrame(yt).head(),
        "yt+1": pd.DataFrame(yt_next).head(),
    },
    axis=1,
)
display(preview)
print(f"yt shape: {yt.shape}")
print(f"yt+1 shape: {yt_next.shape}")

yt                                                              \
          0         1         2         3         4         5         6   
0  0.175146 -0.150038 -0.183743  0.203419  0.232158 -0.116310  0.140070   
1  0.174351 -0.120313 -0.131954  0.169210  0.205579 -0.078307  0.097398   
2  0.167551 -0.132009 -0.121652  0.158200  0.130026 -0.062672  0.086276   
3  0.122673 -0.089351 -0.079237  0.123888  0.126738 -0.031834  0.033574   
4  0.096802 -0.081156 -0.081558  0.118891  0.092831  0.009731  0.018261   

                 yt+1                                                    \
          7         0         1         2         3         4         5   
0  0.085932  0.143415 -0.125130 -0.152896  0.171298  0.210098 -0.067136   
1  0.071258  0.144286 -0.091902 -0.107545  0.146844  0.182564 -0.043402   
2  0.044191  0.131157 -0.107277 -0.095824  0.131411  0.106144 -0.038844   
3  0.047261  0.094958 -0.062026 -0.060580  0.105148  0.105245 -0.014077   
4  0.011807  0.072526 -0.058925 -0.056856  0.097185  0.065512  0.030137   

                       
          6         7  
0  0.097372  0.069981  
1  0.060109  0.049586  
2  0.051978  0.023777  
3  0.004183  0.018219  
4 -0.006726 -0.012086

yt shape: (1000, 8)
yt+1 shape: (1000, 8)
